# TRIBE V2 — Colab pre-render for hackathon demo

Runs `run_inference.py` + `aggregate.py` on the free Colab T4 (16 GB) and writes `activity.json` artifacts that the brain-swarm backend consumes.

**Why this exists:** Vultr GPU deploy is gated behind a $50 deposit. The demo only needs the *output* JSON, not a live GPU — so we pre-render here and ship the JSON with the repo.

**Before you run:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have an HF token ready ([hf.co/settings/tokens](https://huggingface.co/settings/tokens)) with access to `meta-llama/Llama-3.2-3B` and `facebook/tribev2`
3. The repo (junsoo branch) needs to be pushed to GitHub — or upload `junsoo/` manually via the file panel

**Output:** Each video produces `activity.json` (small, ~50–200 KB) plus `preds.npz` + `preds.segments.pkl` (big, ~50 MB). Drive keeps everything; we only commit the JSON to git.

## 1. Sanity check — GPU

In [ ]:
!nvidia-smi

If you see `command not found` or no GPU listed, fix Runtime → Change runtime type before going further.

## 2. Mount Google Drive

Caches HF weights and stores prerendered outputs there so they survive Colab disconnects. The first prefetch is ~15 GB / ~10 min; after that, every session is instant.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/tribe')
HF_CACHE   = DRIVE_ROOT / 'hf_cache'
TRIBE_CACHE = DRIVE_ROOT / 'tribe_cache'   # for atlas labels + tribe demo_utils
OUTPUTS    = DRIVE_ROOT / 'prerendered'
VIDEOS     = DRIVE_ROOT / 'videos'

for p in (HF_CACHE, TRIBE_CACHE, OUTPUTS, VIDEOS):
    p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'hub')

print('Drive paths:')
print(f'  HF cache:    {HF_CACHE}')
print(f'  TRIBE cache: {TRIBE_CACHE}')
print(f'  Outputs:     {OUTPUTS}')
print(f'  Videos:      {VIDEOS}')

## 3. Get the code

**Option A (recommended):** clone from GitHub. Set `REPO_URL` to your fork and the branch you want.

**Option B:** if you haven't pushed `junsoo` branch yet, upload `junsoo/` (with `run_inference.py`, `aggregate.py`, `atlas.py`, `prefetch.sh`, `infer_pipeline.sh`) via the Files panel on the left → drag the folder into `/content/`. Then skip the clone cell.

In [ ]:
REPO_URL = 'https://github.com/<YOUR_GITHUB_USER>/<YOUR_REPO>.git'  # <-- EDIT THIS
BRANCH   = 'junsoo'

!cd /content && git clone --branch {BRANCH} --depth 1 {REPO_URL} hackathon || (cd /content/hackathon && git pull --ff-only)
%cd /content/hackathon/junsoo

## 4. Install TRIBE V2

The repo already has `tribev2/` checked in — install editable with `[plotting]` extras to pull in `nilearn` (atlas).

In [ ]:
!pip install -q --upgrade pip wheel
!pip install -q -e './tribev2[plotting]'

## 5. HuggingFace login

Paste your token when prompted. Needs read access to:
- `meta-llama/Llama-3.2-3B` (gated — request access first at [hf.co/meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B))
- `facebook/tribev2`

In [ ]:
from huggingface_hub import login
login()  # interactive token prompt

## 6. Prefetch weights (one-time, ~15 GB)

First run takes ~10 min. Subsequent sessions skip this — Drive already has the files.

In [ ]:
from pathlib import Path
from tribev2.demo_utils import TribeModel

# This populates HF_CACHE on Drive with all sub-models (Llama, w2v-bert, vjepa2, dinov2)
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=Path(str(TRIBE_CACHE)))
print('TRIBE V2 cached on Drive ✓')

# Pre-build Yeo 7-network atlas labels (also cached on Drive)
from atlas import build_yeo_labels
labels = build_yeo_labels(Path(str(TRIBE_CACHE)), n_networks=7)
print(f'Atlas labels shape: {labels.shape} ✓')

del model  # free VRAM before inference

## 7. Upload demo videos

Three options — pick one:

**A. Drag-drop into `/content/drive/MyDrive/tribe/videos/`** via Drive web UI (most durable; videos persist across sessions)

**B. Upload via Colab files panel** — left sidebar → folder icon → upload to `/content/`. Fast but lost on disconnect.

**C. Download from URL** — fill in below.

In [ ]:
# Option C: download from URL into Drive
VIDEO_URL = ''  # e.g. 'https://example.com/clip1.mp4'
if VIDEO_URL:
    fname = VIDEO_URL.rsplit('/', 1)[-1]
    !wget -q -O '{VIDEOS}/{fname}' '{VIDEO_URL}'
    print(f'Saved to {VIDEOS}/{fname}')

# Show whatever's in the videos dir
!ls -lh '{VIDEOS}/' 2>/dev/null || echo 'No videos yet'

## 8. Run pre-render pipeline

Set `VIDEO_PATH` to one of the videos above. This cell:
1. Calls `run_inference.py` → `preds.npz` + `preds.segments.pkl`
2. Calls `aggregate.py` → `activity.json`
3. Saves all three to `OUTPUTS/<video_stem>/` on Drive

Re-run this cell once per video. ~5–10 min per ~30 s clip on T4.

In [ ]:
VIDEO_PATH = str(VIDEOS / 'clip1.mp4')   # <-- EDIT to a real file from cell 7
ATLAS = 'yeo7'                            # 'yeo7' or 'yeo17'

from pathlib import Path
video = Path(VIDEO_PATH)
assert video.exists(), f'Video not found: {video}'

out_dir = OUTPUTS / video.stem
out_dir.mkdir(parents=True, exist_ok=True)

preds_path = out_dir / 'preds.npz'
json_path  = out_dir / 'activity.json'

print(f'==> Inference: {video.name}')
!python run_inference.py \
    --video '{video}' \
    --out '{preds_path}' \
    --cache '{TRIBE_CACHE}'

print(f'\n==> Aggregating to per-region JSON')
!python aggregate.py \
    --preds '{preds_path}' \
    --segments '{preds_path.with_suffix(".segments.pkl")}' \
    --out '{json_path}' \
    --atlas {ATLAS} \
    --cache '{TRIBE_CACHE}'

print(f'\n==> Done. Outputs in {out_dir}')
!ls -lh '{out_dir}/'

# Quick peek at first frame
import json
with open(json_path) as f:
    d = json.load(f)
print(f'\nFirst frame ({d["n_timesteps"]} total):')
print(json.dumps(d['frames'][0], indent=2))

## 9. Download `activity.json` to your Mac

Two ways:

**A. Download via browser:** Drive web UI → `MyDrive/tribe/prerendered/<video_stem>/activity.json` → right-click → Download.

**B. Use the cell below to package all of `prerendered/` as a zip and download:**

In [ ]:
# Zip and download all activity.json files (skips the big .npz/.pkl)
import shutil, glob
from google.colab import files

tmp = Path('/content/activity_jsons')
if tmp.exists():
    shutil.rmtree(tmp)
tmp.mkdir()

for j in glob.glob(str(OUTPUTS / '*' / 'activity.json')):
    p = Path(j)
    dest = tmp / f'{p.parent.name}_activity.json'
    shutil.copy(j, dest)
    print(f'  {dest.name} ({dest.stat().st_size / 1024:.1f} KB)')

shutil.make_archive('/content/activity_jsons', 'zip', tmp)
files.download('/content/activity_jsons.zip')

## 10. Commit to repo

On your Mac:

```bash
# unzip the downloaded bundle
unzip ~/Downloads/activity_jsons.zip -d junsoo/prerendered/

# OR if you downloaded individual files:
mkdir -p junsoo/prerendered/<video_stem>
cp ~/Downloads/activity.json junsoo/prerendered/<video_stem>/

git add junsoo/prerendered/
git commit -m 'feat(tribe): pre-rendered activity.json for demo videos'
```

Then the brain-swarm backend can read from `junsoo/prerendered/<video>/activity.json` for the demo (or copy to `_bmad/brain-swarm/backend/data/activity.json`).

**Don't commit `preds.npz` or `*.pkl`** — they're 50 MB+ each and `.gitignore` will block them anyway. Drive keeps the originals if you ever need to re-aggregate.

## Troubleshooting

**OOM on T4 (16 GB):** TRIBE V2 should fit, but if it doesn't, try a shorter video clip (chunk to ≤30 s). The model loads in fp16 by default in `demo_utils`.

**`huggingface-cli: 401`:** your token doesn't have access to gated Llama-3.2-3B. Request access on the model page, wait for approval (usually <1 hr), retry.

**Session disconnected mid-render:** outputs up to that point are safe on Drive. Re-run cell 8; `run_inference.py` doesn't checkpoint per-frame so it'll restart from scratch, but the weights are cached.

**No GPU available:** Colab free-tier rotates GPUs by demand. Try off-peak (early AM PT) or sign up for $10/mo Colab Pro.